# Merge the v4-mixed-r16 adapter into PhoWhisper-large and publish

Produces a standalone Whisper checkpoint at `winhsss/Reworkwhisper-large-v5` --
loadable with `WhisperForConditionalGeneration.from_pretrained`, no `peft`, no
`adapter_config.json`.

**No accelerator needed.** The merge runs fp32 on CPU (highest precision; a
published checkpoint is written once and loaded many times). Set the notebook
accelerator to *None* and save the GPU quota.

**Before running:**
- Add `HF_TOKEN` (write scope) under Add-ons -> Secrets.
- Commit and push `scripts/merge_and_push.py` + `experiments/v4-mixed-r16/` to
  GitHub `main` first -- Cell 1 clones `main`, not your local tree.

**Disk:** ~6.2 GB base download into `~/.cache/huggingface` plus ~6.2 GB for the
merged output under `/kaggle/working`. Nothing else should be attached.

## 1. Clone / update the repo

In [ ]:
import os

REPO_URL = "https://github.com/egoist-minh/Reworkwhisper-finetune.git"
REPO_DIR = "/kaggle/working/Reworkwhisper-finetune"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull origin main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt

## 2. Token

Read from this notebook's Secrets, never hardcoded and never written into any
artifact. Needs **write** scope on `winhsss/`.

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["TRANSFORMERS_AUTO_CONVERSION"] = "0"
print("HF_TOKEN set")

## 3. What is in the target repo right now

The merge reads the adapter *from the Hub*. Run `v4-mixed-r16`'s frozen
`config.json` records `hub.push: false`, so the push that wrote
`outputs/v4-mixed-r16/adapter/README.md` happened outside the pipeline and
whether the adapter actually landed in the repo is unverified. Confirm here.

- Lists `adapter_config.json` + `adapter_model.safetensors` -> leave the next
  cell's `ADAPTER` as the repo id.
- Repo missing, or no adapter files -> attach `outputs/v4-mixed-r16/adapter/`
  (110 MB) as a Kaggle Dataset and point `ADAPTER` at its mount path instead.

In [ ]:
from huggingface_hub import HfApi

REPO_ID = "winhsss/Reworkwhisper-large-v5"

try:
    print("\n".join(sorted(HfApi().list_repo_files(REPO_ID))))
except Exception as e:
    print(f"cannot list {REPO_ID}: {type(e).__name__}: {e}")

## 4. Merge (and push once you have read the output)

`CONFIRM_PUSH = False` runs everything except the upload: it verifies provenance
(gate passed, adapter matches this run's base model / rank, baked lambda is the
sweep row the gate scored), merges, checks the merged logits against the
unmerged PeftModel, writes the model card, and stops. Read the printed card and
the `max logit diff` line, then flip the flag and re-run this cell -- the re-run
redoes the merge from scratch (~5 min), it does not reuse the saved folder.

After a successful upload, `--delete-remote-adapter` removes
`adapter_config.json` and `adapter_model.safetensors` from the repo so it ships
the merged model only.

> Deleting those two files from `winhsss/Reworkwhisper-large-v5` is a remote,
> not-undo-in-one-click action. The adapter still exists locally at
> `outputs/v4-mixed-r16/adapter/` and inside `Outputs/outputs_v4-mixed-r16.zip`.
> Drop the `--delete-remote-adapter` flag to keep both in the repo instead.

In [ ]:
import subprocess

CONFIRM_PUSH = False   # flip to True after reading this cell's output yourself

ADAPTER = REPO_ID      # or the mount path of an attached adapter dataset
RUN_DIR = "experiments/v4-mixed-r16"
OUT_DIR = "/kaggle/working/merged-v4-mixed-r16"

cmd = [
    "python", "-m", "scripts.merge_and_push",
    "--run-dir", RUN_DIR,
    "--adapter", ADAPTER,
    "--repo-id", REPO_ID,
    "--out", OUT_DIR,
]
if CONFIRM_PUSH:
    cmd += ["--confirm", "--delete-remote-adapter"]

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
if proc.wait() != 0:
    raise SystemExit(f"merge_and_push failed with exit code {proc.returncode}")

print(open(f"{OUT_DIR}/README.md", encoding="utf-8").read())

## 5. Verify the published repo loads standalone

Downloads the repo fresh (no adapter, no `peft` import) and decodes one silent
frame. This proves the published files are self-sufficient; it says nothing
about CER -- those numbers come from the gate in the model card.

In [ ]:
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor

model = WhisperForConditionalGeneration.from_pretrained(REPO_ID, torch_dtype=torch.float16)
processor = WhisperProcessor.from_pretrained(REPO_ID)
print(sum(p.numel() for p in model.parameters()), "params,", next(model.parameters()).dtype)

silence = torch.zeros(16000, dtype=torch.float32).numpy()
feats = processor(silence, sampling_rate=16000, return_tensors="pt").input_features.half()
ids = model.generate(feats, language="vi", task="transcribe", max_new_tokens=16)
print(repr(processor.batch_decode(ids, skip_special_tokens=True)[0]))